# Qwen-Image-2512 Character LoRA — Google Colab Training

Train a character-specific LoRA on Qwen-Image-2512 (20B MMDiT) directly in Google Colab.

**Recommended runtime:** A100 (80 GB) or H100 — minimum 40 GB VRAM required. T4 is insufficient.

Run cells top-to-bottom. Every section is idempotent — if your runtime disconnects, reconnect, re-run all cells, and training resumes from the latest checkpoint automatically.

## Before You Start

### 1. Colab Secret: `HF_TOKEN`
Go to **Secrets** (key icon in the left sidebar) and add:
- Name: `HF_TOKEN`
- Value: your HuggingFace access token with read access to `Qwen/Qwen-Image-2512`
- Enable **Notebook access**

### 2. Google Drive: training dataset
Put your captioned subject images zip at:
```
MyDrive/lora-trainer/input/batch1.zip
```
The zip should contain images + matching `.txt` caption files (one `.txt` per image, same filename).
Use `scripts/caption.py` from the repo to generate captions if you haven't yet.

### Directory layout (for reference)
```
/content/
├── drive/MyDrive/lora-trainer/
│   ├── input/
│   │   └── batch1.zip          ← YOUR DATASET (put here before running)
│   └── output/
│       └── qwen-batch1/        ← LoRA weights saved here (persists after disconnect)
│
└── data/                       ← ephemeral working dir (lost on runtime reset)
    ├── models/Qwen-Image-2512/ ← ~40 GB base model
    ├── dataset/
    │   ├── images/             ← your subject images + .txt captions
    │   └── regularization/
    │       └── images/         ← regularization dataset (auto-downloaded)
    ├── cache/                  ← SimpleTuner VAE + text embed cache
    └── config/                 ← generated config files
```

In [ ]:
# ── User Config ────────────────────────────────────────────────────────────────
# Edit these values before running the notebook.

TRIGGER_WORD        = "ohwx"          # Trigger word embedded in all captions
LORA_OUTPUT_NAME    = "qwen-batch1"   # Subfolder name under Drive output dir
MAX_TRAIN_STEPS     = 3000
TRAIN_BATCH_SIZE    = 1               # A100 80 GB can handle 2; H100 can handle 2-4
LORA_RANK           = 64
LORA_ALPHA          = 64
LEARNING_RATE       = 3e-5

# Attention mechanism — choose based on your GPU:
#   "flash_attention_2"  → A100, H100 (recommended for Colab)
#   "sdpa"               → any GPU, safe fallback with lower throughput
#   "flash-attn-3"       → Blackwell only (GB10/GB200), NOT available on Colab
ATTENTION_MECHANISM = "flash_attention_2"

# Paths on Google Drive
DATASET_ZIP_DRIVE_PATH = f"/content/drive/MyDrive/lora-trainer/input/batch1.zip"
OUTPUT_DRIVE_DIR       = f"/content/drive/MyDrive/lora-trainer/output/{LORA_OUTPUT_NAME}"

# Internal working paths (ephemeral, under /content/data)
DATA_ROOT            = "/content/data"
MODEL_DIR            = f"{DATA_ROOT}/models/Qwen-Image-2512"
SUBJECT_IMAGES_DIR   = f"{DATA_ROOT}/dataset/images"
REG_IMAGES_DIR       = f"{DATA_ROOT}/dataset/regularization/images"
CONFIG_DIR           = f"{DATA_ROOT}/config"
VAE_CACHE_SUBJECT    = f"{DATA_ROOT}/cache/vae/subject"
VAE_CACHE_REG        = f"{DATA_ROOT}/cache/vae/regularization"
TEXT_CACHE_DIR       = f"{DATA_ROOT}/cache/text/qwen_image"
# ──────────────────────────────────────────────────────────────────────────────

print("Config loaded.")
print(f"  Output name : {LORA_OUTPUT_NAME}")
print(f"  Train steps : {MAX_TRAIN_STEPS}")
print(f"  Batch size  : {TRAIN_BATCH_SIZE}")
print(f"  Attention   : {ATTENTION_MECHANISM}")

## Section 0: Runtime Check
Verify a GPU is attached and print its specs.

In [ ]:
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU found. Change runtime type to GPU (A100 or H100).")

gpu_info = result.stdout.strip()
print(f"GPU: {gpu_info}")

import torch
assert torch.cuda.is_available(), "CUDA not available — check runtime type"
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

gpu_name = gpu_info.split(',')[0].strip().lower()
if not any(g in gpu_name for g in ['a100', 'h100', 'h200']):
    print(f"\nWARNING: GPU '{gpu_info.split(chr(44))[0].strip()}' may not have enough VRAM.")
    print("Recommended: A100 (80 GB) or H100. Minimum 40 GB VRAM required.")
else:
    print("GPU OK.")

## Section 1: Mount Google Drive
Mounts Drive and verifies your dataset zip is present.

In [ ]:
import os
from pathlib import Path

# Mount Drive (idempotent)
if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Drive already mounted.")

# Verify dataset zip exists
zip_path = Path(DATASET_ZIP_DRIVE_PATH)
if not zip_path.exists():
    raise FileNotFoundError(
        f"Dataset zip not found at {zip_path}\n"
        "Please upload batch1.zip to MyDrive/lora-trainer/input/ and re-run."
    )
print(f"Dataset zip found: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")

# Create output directory on Drive
Path(OUTPUT_DRIVE_DIR).mkdir(parents=True, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DRIVE_DIR}")

### Optional: Cache model weights on Drive

The base model (~40 GB) is re-downloaded every time you get a fresh runtime. If you want to avoid this, run the cell below **once** to symlink the model directory to Google Drive. Subsequent runtimes will find the weights already there.

**Trade-off:** Drive I/O is slower than Colab's local SSD. Training startup will take a few extra minutes.

**Skip this cell if you prefer a fresh download each time (faster during training).**

In [ ]:
# OPTIONAL — run this cell to cache the model on Drive
# Skip entirely if you want the model on fast local SSD instead.

import os
from pathlib import Path

drive_model_cache = "/content/drive/MyDrive/lora-trainer/model-cache/Qwen-Image-2512"
Path(drive_model_cache).mkdir(parents=True, exist_ok=True)

local_model_parent = Path(MODEL_DIR).parent
local_model_parent.mkdir(parents=True, exist_ok=True)

if Path(MODEL_DIR).is_symlink():
    print(f"Symlink already exists: {MODEL_DIR} -> {os.readlink(MODEL_DIR)}")
elif Path(MODEL_DIR).exists():
    print(f"{MODEL_DIR} already exists as a real directory (not symlinking).")
else:
    os.symlink(drive_model_cache, MODEL_DIR)
    print(f"Symlinked {MODEL_DIR} -> {drive_model_cache}")
    print("Model weights will be cached to Drive and reused across runtimes.")

## Section 2: Install Dependencies
Installs SimpleTuner from git main (required for Qwen-Image batch>1 fix). Skips if already installed.

In [ ]:
import importlib.util, subprocess, sys

def run(cmd, **kwargs):
    """Run a shell command, streaming output, raising on failure."""
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

# Install SimpleTuner if not already present
if importlib.util.find_spec("simpletuner") is None:
    print("Installing SimpleTuner from git main...")
    run("pip install -q 'simpletuner[cuda] @ git+https://github.com/bghira/SimpleTuner.git@main'")
    print("SimpleTuner installed.")
else:
    print("SimpleTuner already installed, skipping.")

# Apply safety_check.py patch (handles nvidia-smi returning [N/A] on unified-memory GPUs).
# On Colab discrete GPUs this is a no-op functionally, but keeps parity with the Dockerfile.
import simpletuner, os
safety_check_path = os.path.join(
    os.path.dirname(simpletuner.__file__),
    "helpers", "training", "default_settings", "safety_check.py"
)
old_line = "total_memory = int(output.decode().strip()) / 1024"
new_line  = "raw = output.decode().strip(); total_memory = (int(raw) if raw.lstrip('-').isdigit() else 131072) / 1024"

with open(safety_check_path, 'r') as f:
    content = f.read()

if old_line in content:
    with open(safety_check_path, 'w') as f:
        f.write(content.replace(old_line, new_line))
    print("safety_check.py patched.")
else:
    print("safety_check.py already patched or line not found (skipping).")

print("Dependencies ready.")